# ЛР2 — практика B: самостоятельная работа с транзакциями

**Тема:** несколько таблиц, масштаб, `merge`, `groupby`, `pivot_table`, `qcut` и feature table  
**Формат:** самостоятельная часть второй практики ЛР2  
**Контроль:** одна общая защита после A+B

Вы работаете с четырьмя связанными таблицами и 100 000 транзакций. Цель — не «применить как можно больше методов pandas», а получить корректную таблицу уровня клиента и уметь доказать, что при объединениях не потерялись и не размножились данные.

## 0. Правила работы

- Сначала словами укажите, что означает одна строка каждой таблицы.
- Перед каждым `merge` запишите ожидаемый тип связи.
- Используйте `validate` там, где ожидание можно проверить автоматически.
- После важных трансформаций проверяйте число строк и уникальность ключа.
- Не заполняйте любой `NaN` нулём автоматически: сначала объясните его смысл.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

candidate_dirs = [
    Path("../data"),
    Path("data"),
    Path("../../../shared/02_tabular_data/data"),
]
DATA_DIR = next((p for p in candidate_dirs if (p / "transactions.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Не найден каталог данных ЛР2.")

print("DATA_DIR:", DATA_DIR.resolve())

# 1. Загрузка и смысл таблиц

Загрузите:

- `clients.csv` с `parse_dates=["registration_date"]`;
- `transactions.csv` с `parse_dates=["transaction_time"]`;
- `mcc_codes.csv`;
- `transaction_types.csv`.

Выведите `shape`, первые строки и типы столбцов. Затем одной фразой зафиксируйте смысл строки каждой таблицы.

In [ ]:
# TODO: загрузите четыре таблицы и выполните первичную диагностику.

# 2. Задание 1 — аудит ключей и масштаба

Составьте функцию `key_report(df, key)`, которая возвращает:

- число строк;
- число уникальных ключей;
- число строк, входящих в повторяющиеся значения ключа.

Примените её к ключам:

- `clients.client_id`;
- `transactions.transaction_id`;
- `transactions.client_id`;
- `mcc_codes.mcc_code`;
- `transaction_types.transaction_type_id`.

После таблицы напишите, где повтор ключа является ошибкой, а где — ожидаемым свойством данных.

In [ ]:
# TODO: реализуйте key_report и соберите сводную таблицу.

# 3. Задание 2 — обогащение транзакций через безопасный `merge`

Соберите таблицу `tx`, в которой каждая строка всё ещё означает **одну транзакцию**, но дополнительно есть:

- `category` из MCC-справочника;
- `transaction_type` и `direction` из справочника типов;
- `city` и `segment` клиента.

Требования:

1. везде использовать `how="left"`;
2. указать корректный `validate`;
3. при merge MCC включить `indicator` и посчитать `left_only`;
4. показать несколько неприсоединившихся MCC;
5. заменить отсутствующую `category` на строку `"Unknown"`;
6. проверить, что число строк `tx` равно числу исходных транзакций.

In [ ]:
# TODO: соберите tx и выполните проверки.

# 4. Задание 3 — извлекаем признаки времени

Из `transaction_time` создайте:

- `month` — номер месяца;
- `weekday` — название дня недели;
- `hour` — час операции.

Затем найдите:

1. пять часов с наибольшим числом транзакций;
2. по каждому `weekday` — число операций и сумму `amount`.

В 2–3 предложениях объясните, почему `parse_dates` лучше сделать при чтении CSV, а не хранить дату обычной строкой.

In [ ]:
# TODO: временные признаки и две агрегации.

# 5. Задание 4 — `groupby + agg`: одна строка на клиента

Постройте три агрегированные таблицы.

### `base_features`
- `transaction_count`;
- `active_days` — число уникальных календарных дней с операциями;
- `unique_categories`.

### `expense_features` только для `direction == "expense"`
- `expense_count`;
- `total_expense`;
- `avg_expense`;
- `cash_withdrawal_count` — число операций типа `Cash withdrawal`.

### `income_features` только для `direction == "income"`
- `income_count`;
- `total_income`.

Проверьте, что `client_id` уникален во всех трёх агрегатах.

In [ ]:
# TODO: base_features, expense_features, income_features.

# 6. Задание 5 — `pivot_table`: расходы по категориям

Для расходных операций постройте `pivot_table`:

- строки — `client_id`;
- столбцы — `category`;
- значения — сумма `amount`;
- пропуски заполнить нулём.

Переименуйте получившиеся столбцы в формат `spend_<category>`, предварительно приведя название категории к нижнему регистру и заменив пробелы на `_`.

Это уже не «отчётная сводная таблица», а набор признаков, который затем можно присоединить к клиенту.

In [ ]:
# TODO: category_spend = tx_expense.pivot_table(...) и подготовка имён столбцов.

# 7. Задание 6 — собираем итоговую `features`

Начните с **полного** `clients`, затем последовательно добавьте:

- `base_features`;
- `expense_features`;
- `income_features`;
- `category_spend`.

Используйте `how="left"` и `validate="one_to_one"`.

После merge:

- счётчики и суммы для клиентов без операций заполните нулями;
- `avg_expense` **не заполняйте нулём автоматически** — оставьте `NaN` и в комментарии объясните почему;
- проверьте, сколько клиентов вообще не имеет транзакций.

In [ ]:
# TODO: соберите features и осмысленно обработайте NaN.

# 8. Задание 7 — `qcut`: сегментация по расходам

Для клиентов с `total_expense > 0` разбейте `total_expense` на четыре группы одинаковой численности с помощью `pd.qcut`:

`low`, `medium`, `high`, `very_high`.

Добавьте результат в столбец `spend_group`. Для клиентов без расходов оставьте отдельную категорию `no_expense`.

Выведите число клиентов в каждой группе и медиану `total_expense` по группам.

In [ ]:
# TODO: spend_group через pd.qcut.

# 9. Задание 8 — sanity checks как часть pipeline

Соберите `pd.Series` минимум из следующих проверок:

- `client_id` уникален в `features`;
- число строк равно `len(clients)`;
- набор клиентов полностью сохранён;
- сумма `transaction_count` равна числу исходных транзакций;
- сумма `expense_count + income_count` равна числу транзакций;
- `total_expense` и `total_income` неотрицательны;
- нет пропусков в `spend_group`.

Все проверки должны дать `True`.

In [ ]:
# TODO: sanity = pd.Series({...})

# 10. ⭐ Дополнительное задание — одна гипотеза, а не каталог графиков

Выберите **одну** гипотезу и проверьте её агрегатами или одним графиком. Например:

- отличаются ли категории расходов между `premium` и `regular`;
- различается ли доля cash withdrawal по городам;
- отличается ли активность по часам у разных каналов.

До расчёта сформулируйте ожидаемый результат. После расчёта напишите 3–5 предложений: что получилось, что можно утверждать по этим данным и чего утверждать нельзя.

In [ ]:
# TODO (⭐): одна выбранная гипотеза + расчёт/график + краткий вывод.

# 11. Итог и подготовка к общей защите ЛР2

Ответьте 7–10 предложениями:

1. Что означает «уровень наблюдения» и почему он важен при `merge`?
2. Почему `client_id` может быть уникальным в одной таблице и повторяться в другой?
3. Что проверяет `validate="many_to_one"`?
4. Почему прямой many-to-many merge событий может испортить агрегаты?
5. Зачем анализировать `left_only` после соединения со справочником?
6. Почему клиента без операций разумно сохранить через `left join`?
7. Почему для такого клиента `transaction_count=0`, но `avg_expense=NaN` может быть осмысленнее, чем `0`?
8. Что даёт `qcut`, чего не даёт фиксированная граница вроде `total_expense > 100000`?

На защите важно не только показать готовый DataFrame, но и уметь объяснить, **почему число строк и смысл строки после каждого ключевого шага именно такие**.